# **C13 - Black Scholes para valuar opciones europeas**
## **Simulación de Procesos Financieros**

---

**Autor:** Francisco Uriel Ledezma Chávez  
**Carrera:** Ingeniería Financiera  
**Institución:** ITESO — Universidad Jesuita de Guadalajara  
**Semestre:** 5° — Simulación de Procesos Financieros  
**Fecha de realización:** 19 de octubre de 2026

---

**Ejercicio:** comprobar la simetría de la normal estándar, valuar una call europea con $S_0 = 100$, $K = 100$, $r = 5\%$, $\sigma = 20\%$ y $T = 1$ a partir de un valor de $Z$ y de su opuesto, y repetir el cálculo con los diez valores de $Z$ dados en clase para comparar el promedio de cada par antitético contra la fórmula de Black-Scholes.

C12 redujo la varianza apoyándose en un contrato parecido cuyo precio se conoce por forma cerrada, mientras que las variables antitéticas no necesitan ningún pariente analítico, ya que aprovechan una propiedad de la propia normal estándar: si $Z$ es una extracción válida, $-Z$ también lo es y tiene la misma probabilidad, de modo que cada número aleatorio puede usarse dos veces y el par resultante tiende a compensar sus errores, porque cuando un escenario empuja el precio hacia arriba su gemelo lo empuja en la misma magnitud hacia abajo.

## Índice de contenidos

1. Sección 1 — Simetría de la normal estándar
2. Sección 2 — Parámetros y precio de referencia
3. Sección 3 — Un par antitético
4. Sección 4 — Diez pares antitéticos
5. Sección 5 — Valor presente y precisión de los diez pares
6. Sección 6 — Diez mil pares frente a Monte Carlo simple
7. Sección 7 — Conclusiones

## Importación de bibliotecas

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(23)

## 1. Simetría de la normal estándar

El método descansa en que la normal estándar es simétrica alrededor de cero, de modo que antes de usar esa propiedad conviene verificarla sobre una muestra de 10,000 extracciones, contando qué fracción cae a cada lado del centro.

In [2]:
Distribucion_normal = np.random.normal(0, 1, 10000)

In [3]:
Distribucion_normal_positiva = (Distribucion_normal > 0).mean()
Distribucion_normal_negativa = (Distribucion_normal < 0).mean()
Distribucion_normal_positiva, Distribucion_normal_negativa

(np.float64(0.4925), np.float64(0.5075))

La muestra reparte 49.25% de las extracciones por encima de cero y 50.75% por debajo, una diferencia de 1.5 puntos que corresponde al ruido esperado de una muestra finita, ya que la desviación estándar de una proporción de 0.5 con 10,000 observaciones es de 0.5 puntos porcentuales, y precisamente ese desbalance es el que la técnica antitética elimina, pues al sumar $-Z$ a cada $Z$ la muestra queda simétrica por construcción y no por azar.

## 2. Parámetros y precio de referencia

In [4]:
S0 = 100      # Precio inicial
k = 100       # Precio de ejercicio
std = 0.2     # Volatilidad
r = 0.05      # Tasa libre de riesgo
T = 1         # Vencimiento en años
M = 10000     # Número de pares antitéticos de la sección 6

La referencia es la fórmula de Black-Scholes, que para la call europea entrega el precio exacto bajo los mismos supuestos de la simulación, y se calcula con las funciones `d1_d2` y `bs` heredadas de C12, cuyo argumento `q` es el costo de acarreo y no un dividendo, de modo que la europea sin dividendos corresponde a $q = r$.

$$C = S_0 N(d_1) - K e^{-rT} N(d_2), \qquad d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)\,T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

In [5]:
def d1_d2(S, K, r, sigma, T, q):
    d1 = (np.log(S / K) + (q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return d1, d1 - sigma * np.sqrt(T)


def bs(S, K, r, sigma, T, call=True, q=0):
    """Black-Scholes con costo de acarreo q; con q = r es la europea sin dividendos. Usa d1_d2."""
    d1, d2 = d1_d2(S, K, r, sigma, T, q)
    if call:
        return S * np.exp((q - r) * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp((q - r) * T) * norm.cdf(-d1)

In [6]:
d1, d2 = d1_d2(
    S0, k, r, std, T, q=r
)

call_europea = bs(S0, k, r, std, T, call=True, q=r)

pd.Series({
    "d1": d1,
    "d2": d2,
    "Call europea (referencia)": call_europea,
}, name="Valor").round(4).to_frame()

,Valor
d1,0.3500
d2,0.1500
Call europea (referencia),10.4506


Con $q = r$ los dos parámetros valen $d_1 = 0.35$ y $d_2 = 0.15$, de modo que la call europea vale 10.4506, cifra que funciona como valor exacto para todo lo que sigue, porque cualquier distancia entre ella y una estimación simulada proviene únicamente del muestreo y no de una diferencia entre modelos.

## 3. Un par antitético

El precio terminal del movimiento browniano geométrico depende de un solo número aleatorio, y la idea antitética consiste en evaluarlo dos veces, una con $Z$ y otra con $-Z$, comenzando por el caso más simple, $Z = 1$ y su opuesto $Z = -1$.

$$S_T(\pm Z) = S_0 \exp\left[\left(r - \frac{\sigma^2}{2}\right)T \pm \sigma\sqrt{T}\,Z\right], \qquad f(\pm Z) = \max\left(S_T(\pm Z) - K,\ 0\right)$$

In [7]:
V_z = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * 1) - k, 0)
St_z1 = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * 1), 0)
V_z, St_z1

(np.float64(25.86000099294779), np.float64(125.86000099294779))

In [8]:
V_z_antitetica = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * -1) - k, 0)
St_z1 = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * -1), 0)
V_z_antitetica, St_z1

(np.float64(0.0), np.float64(84.36648165963837))

Con $Z = 1$ el precio termina en 125.86 y la call paga 25.86, mientras que con $Z = -1$ el precio cae a 84.37 y la opción vence sin valor, de modo que el par aporta un pago promedio de 12.93, y la asimetría entre ambos movimientos, 25.86 hacia arriba frente a 15.63 hacia abajo, no es un error sino la forma lognormal del precio, que convierte desplazamientos simétricos de $Z$ en desplazamientos asimétricos de $S_T$.

## 4. Diez pares antitéticos

Los diez valores de $Z$ dados en clase repiten el mismo procedimiento a una escala que ya permite compararlo contra Black-Scholes, calculando para cada uno el pago con $Z$, el pago con $-Z$ y el promedio de ambos, que es el pago antitético del par.

In [9]:
Z = np.array([0.07629, -0.2985, -0.7983, 0.993, -1.3465, 1.2462, -1.1405, 0.7615, 0.2816, 0.81])
print(f"Media de los diez valores de Z: {Z.mean():.4f}")
print(f"Media de Z junto con -Z:        {np.concatenate([Z, -Z]).mean():.4f}")

Media de los diez valores de Z: 0.0585
Media de Z junto con -Z:        0.0000


In [10]:
V_z = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * Z) - k, 0)
St_z1 = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * Z), 0)
V_z, St_z1

(array([ 4.62977699,  0.        ,  0.        , 25.68392028,  0.        ,
        32.21246157,  0.        , 19.99741321,  9.01551221, 21.1670517 ]),
 array([104.62977699,  97.07367109,  87.83940341, 125.68392028,
         78.71786933, 132.21246157,  82.02878203, 119.99741321,
        109.01551221, 121.1670517 ]))

In [11]:
V_z_antitetica = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * -Z) - k, 0)
St_z1 = np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * -Z), 0)
V_z_antitetica, St_z1

(array([ 1.48511992,  9.38460807, 20.88385227,  0.        , 34.8914237 ,
         0.        , 29.4468259 ,  0.        ,  0.        ,  0.        ]),
 array([101.48511992, 109.38460807, 120.88385227,  84.48467745,
        134.8914237 ,  80.31289441, 129.4468259 ,  88.48828638,
         97.40233523,  87.63409951]))

In [12]:
Values = pd.DataFrame({'V_z': V_z, 'V_z_antitetica': V_z_antitetica}, index = Z)
Values['Promedio'] = Values[['V_z', 'V_z_antitetica']].mean(axis=1)

print("Promedio de V_z:", Values['V_z'].mean())
print("Promedio de V_z_antitetica:", Values['V_z_antitetica'].mean())
print("Promedio de V_z y V_z_antitetica:", Values['Promedio'].mean())

Values

Promedio de V_z: 11.270613596100127
Promedio de V_z_antitetica: 9.609182985878721
Promedio de V_z y V_z_antitetica: 10.439898290989424


,V_z,V_z_antitetica,Promedio
0.07629,4.629777,1.485120,3.057448
-0.29850,0.000000,9.384608,4.692304
-0.79830,0.000000,20.883852,10.441926
0.99300,25.683920,0.000000,12.841960
-1.34650,0.000000,34.891424,17.445712
1.24620,32.212462,0.000000,16.106231
-1.14050,0.000000,29.446826,14.723413
0.76150,19.997413,0.000000,9.998707
0.28160,9.015512,0.000000,4.507756
0.81000,21.167052,0.000000,10.583526


La media de los diez valores de $Z$ es de 0.0585 y no de cero, y ese pequeño sesgo hacia arriba se refleja en los pagos, ya que el promedio con $Z$ es de 11.2706 y el promedio con $-Z$ de 9.6092, mientras que el conjunto ampliado con los opuestos tiene media exactamente cero y su promedio de pagos, 10.4399, queda entre ambos.

La tabla muestra además el mecanismo del método, pues en nueve de los diez pares sólo uno de los dos escenarios termina dentro del dinero, como en $Z = -1.3465$, donde el pago directo es nulo y el antitético alcanza 34.8914, y la única excepción es $Z = 0.07629$, tan cercano a cero que ambos precios terminan apenas por encima del strike, en 104.63 y 101.49, con lo cual cada par mezcla casi siempre un escenario favorable con uno desfavorable y su promedio varía mucho menos que cualquiera de sus dos componentes.

La cifra de 10.4399 parece coincidir con los 10.4506 de Black-Scholes, aunque esa cercanía es engañosa, porque la tabla promedia pagos al vencimiento mientras que Black-Scholes entrega un valor presente, de modo que la comparación válida exige descontar antes el pago promedio, que es lo que hace la sección siguiente.

## 5. Valor presente y precisión de los diez pares

El precio de Monte Carlo es el pago promedio descontado a la tasa libre de riesgo, y junto a él se reporta su error estándar, de modo que puede compararse la estimación hecha sólo con los diez valores de $Z$ contra la hecha con los diez pares antitéticos.

$$\hat{C}_{AV} = e^{-rT}\,\frac{1}{M}\sum_{i=1}^{M}\frac{f(Z_i) + f(-Z_i)}{2}, \qquad \text{EE} = e^{-rT}\,\frac{s}{\sqrt{M}}$$

In [13]:
descuento = np.exp(-r * T)


def precio_y_error(pagos):
    # Precio de Monte Carlo y su error estándar, ambos ya descontados
    return descuento * np.mean(pagos), descuento * np.std(pagos, ddof=1) / np.sqrt(len(pagos))


precio_simple, ee_simple = precio_y_error(Values['V_z'])
precio_antitetico, ee_antitetico = precio_y_error(Values['Promedio'])

print(f"Call europea por Black-Scholes:            {call_europea:.4f}")
print(f"Correlación entre V_z y V_z_antitetica:    {np.corrcoef(Values['V_z'], Values['V_z_antitetica'])[0, 1]:.4f}")
print(f"Pares en los que sólo uno de los dos paga: {((Values['V_z'] > 0) ^ (Values['V_z_antitetica'] > 0)).sum()} de {len(Values)}")

pd.DataFrame({
    "Precio": [precio_simple, precio_antitetico],
    "Error frente a Black-Scholes": [precio_simple - call_europea, precio_antitetico - call_europea],
    "Error estándar": [ee_simple, ee_antitetico],
}, index=["Diez valores de Z", "Diez pares antitéticos"]).round(4)

Call europea por Black-Scholes:            10.4506
Correlación entre V_z y V_z_antitetica:    -0.7058
Pares en los que sólo uno de los dos paga: 9 de 10


,Precio,Error frente a Black-Scholes,Error estándar
Diez valores de Z,10.7209,0.2704,3.7217
Diez pares antitéticos,9.9307,-0.5198,1.5148


Descontados, los diez valores de $Z$ estiman la call en 10.7209, 0.2704 por encima de Black-Scholes, y los diez pares antitéticos en 9.9307, 0.5198 por debajo, de modo que en esta muestra la técnica antitética termina más lejos del valor exacto, y la coincidencia aparente de la sección anterior desaparece en cuanto se compara un valor presente contra otro.

Ese resultado no contradice el método, ya que una sola muestra puede acertar o fallar por azar, como mostró C11, y la medida de precisión es el error estándar, que cae de 3.7217 a 1.5148, 2.46 veces menor, gracias a una correlación de −0.7058 entre los pagos de cada par, aunque el par evalúa dos pagos por cada valor de $Z$, de modo que la comparación justa, a igual número de pagos y con una muestra en la que el azar pese menos, es la de la sección siguiente.

## 6. Diez mil pares frente a Monte Carlo simple

La comparación justa entre ambos métodos debe hacerse a igual costo, por lo cual se simulan 10,000 valores de $Z$ con sus opuestos, que suman 20,000 pagos, frente a 20,000 valores independientes valuados por Monte Carlo simple, y como el pago de la call es monótono en $Z$, la teoría anticipa que la varianza se reduzca en un factor de $1/(1+\rho)$, donde $\rho$ es la correlación entre el pago de $Z$ y el de $-Z$.

In [14]:
def pago_call(z):
    return np.maximum(S0 * np.exp((r - 0.5 * std ** 2) * T + std * np.sqrt(T) * z) - k, 0)


z = np.random.standard_normal(M)
pagos_pares = (pago_call(z) + pago_call(-z)) / 2

z_independientes = np.random.standard_normal(2 * M)
pagos_simples = pago_call(z_independientes)

precio_mc, ee_mc = precio_y_error(pagos_simples)
precio_av, ee_av = precio_y_error(pagos_pares)

print(f"Correlación entre el pago de z y el de -z: {np.corrcoef(pago_call(z), pago_call(-z))[0, 1]:.4f}")
print(f"Reducción de varianza a igual número de pagos: {(ee_mc / ee_av) ** 2:.3f} veces")

pd.DataFrame({
    "Pagos evaluados": [2 * M, 2 * M],
    "Precio": [precio_mc, precio_av],
    "Error frente a Black-Scholes": [precio_mc - call_europea, precio_av - call_europea],
    "Error estándar": [ee_mc, ee_av],
}, index=["Monte Carlo simple", "Pares antitéticos"]).round(4)

Correlación entre el pago de z y el de -z: -0.4930
Reducción de varianza a igual número de pagos: 1.887 veces


,Pagos evaluados,Precio,Error frente a Black-Scholes,Error estándar
Monte Carlo simple,20000,10.3188,-0.1318,0.1025
Pares antitéticos,20000,10.4374,-0.0132,0.0746


Con el mismo número de pagos, Monte Carlo simple entrega 10.3188 con un error estándar de 0.1025, mientras que los pares antitéticos entregan 10.4374 con un error estándar de 0.0746, apenas 0.0132 por debajo de Black-Scholes, y la reducción de varianza resulta de 1.887 veces, cercana a las 1.97 que anticipa la correlación observada de −0.4930.

La correlación es menor en magnitud que la de los diez pares de clase, que alcanzaba −0.7058 por azar de la muestra, y está acotada por la forma del pago, ya que una call no responde de manera lineal a $Z$, pues su pago se anula en cuanto el precio cae por debajo del strike, de modo que en un par como $Z = -1.3465$ el escenario favorable paga 34.8914 mientras que el desfavorable no puede restar nada por debajo de cero, y el promedio del par conserva así la mitad de la variación de su escenario ganador, por lo cual la reducción, aunque real, queda lejos de los tres órdenes de magnitud que obtuvo la variable de control de C12.

## 7. Conclusiones

El ejercicio confirma que las variables antitéticas reducen la varianza del precio de una call europea sin necesidad de ninguna forma cerrada auxiliar, ya que con 20,000 pagos el error estándar baja de 0.1025 con Monte Carlo simple a 0.0746 con pares antitéticos, una reducción de varianza de 1.887 veces que equivale a obtener la precisión de unos 37,700 pagos independientes con sólo 20,000.

La explicación está en la simetría de la normal estándar, que hace de $-Z$ una extracción tan válida como $Z$ y produce pares con correlación negativa, de −0.4930 en la muestra grande, de modo que el promedio de cada par compensa parte del error de sus dos escenarios, mientras que la forma del pago limita esa compensación, porque el piso en cero de la call impide que el escenario desfavorable reste tanto como suma el favorable.

Los diez pares de clase dejan una advertencia que conviene no perder, pues su promedio sin descontar de 10.4399 parecía reproducir los 10.4506 de Black-Scholes, aunque descontado vale 9.9307 y queda 0.5198 por debajo, con lo cual la cercanía era una coincidencia entre un pago al vencimiento y un valor presente, y la precisión de una muestra tan pequeña debe leerse en su error estándar de 1.5148 y no en su distancia casual a la cifra exacta.

La consecuencia práctica es que la técnica antitética conviene como primer recurso por su costo casi nulo, ya que sólo exige reutilizar cada número aleatorio con el signo contrario, aunque su beneficio depende de qué tan lineal sea el pago frente a $Z$, y para contratos con un pariente de forma cerrada, como la asiática geométrica de C12, la variable de control ofrece reducciones mucho mayores, de modo que ambas técnicas resultan complementarias y pueden combinarse sobre la misma simulación.

El análisis tiene un límite que conviene declarar, ya que la reducción de 1.887 veces proviene de una sola corrida y, como mostró C12 al comparar su factor estimado contra el efectivo, ese cociente debe medirse sobre repeticiones antes de darse por definitivo, de modo que la extensión natural es repetir el experimento varias veces y llevar la misma técnica a contratos que dependen del camino, donde cada trayectoria simulada tiene también su trayectoria antitética.